In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import scipy.stats as stats
from scipy.stats import spearmanr, hypergeom

from statsmodels.stats.multitest import multipletests

# GO enrichment
import gseapy as gp

## CellOracle
import celloracle as co

In [2]:
## DATA (CellOracle object before fitting GRN)
oracle = co.load_hdf5("../data/celloracle_data/celloracle_unfit.celloracle.oracle")

## FIT GRN — RIDGE REGRESSION PER CLUSTER

This estimates the actual regulatory weights W_ij from expression co-variation, using oracle.TFdict as sparsity mask.

==> alpha controls Ridge penalization (higher = sparser weights)

In [ ]:
# Step 1: infer GRN per cluster with Ridge Regression
# alpha: regularization strength. Higher = sparser network.
# verbose_level=10 prints progress per cluster
links = oracle.get_links(
    cluster_name_for_GRN_unit='leiden',
    alpha=10,
    verbose_level=10
)

# Step 2: inspect raw edge distribution before filtering
links.plot_degree_distributions(
    plot_model=True,
    save=None
)

# Step 3: filter edges
# p: maximum adjusted p-value for the Ridge coefficient
# weight: rank edges by absolute coefficient value
# threshold_number: keep top N edges per cluster (not total)
links.filter_links(
    p=0.001,
    weight='coef_abs',
    threshold_number=2000
)

# Optional: check how many edges survive per cluster
links.links_dict  # dict {cluster_id: DataFrame with filtered edges}
for cluster, df in links.links_dict.items():
    print(f"Cluster {cluster}: {len(df)} edges after filtering")

# Step 4: use filtered links for simulation
oracle.get_cluster_specific_TFdict_and_TFMatrix(
    links_object=links
)
oracle.fit_GRN_for_simulation(
    alpha=10,
    use_cluster_specific_TFdict=True  # uses the filtered per-cluster networks
)

KeyError: 'draw_graph'

## Simulate shift

Also extracts expected expression shift after KO

In [ ]:
# =============================================================
# STEP 6: SIMULATE RMST1 KNOCKOUT
# Sets Rmst expression to 0 in all cells and propagates
# the perturbation through the fitted GRN.
# =============================================================

oracle.simulate_shift(
    perturb_condition={"Rmst": 0.0},  # KO: set Rmst to zero
    n_propagation=3   # number of propagation steps through the network
)


# =============================================================
# STEP 7: COMPUTE TRANSITION PROBABILITIES AND EMBEDDING SHIFT
# Translates the gene expression shift into a probability of
# transitioning to neighboring cells in the UMAP embedding.
# =============================================================

oracle.estimate_transition_prob(
    n_neighbors=40,
    knn_random=True,
    sampled_fraction=0.5
)

oracle.calculate_embedding_shift(sigma_corr=0.05)


# =============================================================
# STEP 8: VISUALIZE VECTOR FIELD ON UMAP
# =============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Grid-based vector field (cleaner visualization)
oracle.plot_simulation_flow_on_grid(
    scale=0.4,
    ax=axes[0],
)
axes[0].set_title('RMST1 KO — vector field (grid)')

# Single-cell arrows
oracle.plot_simulation_flow_random_sampling(
    scale=0.4,
    ax=axes[1],
    color_by='leiden',
    n_each_cluster=30
)
axes[1].set_title('RMST1 KO — vector field (single cells)')

plt.tight_layout()
#plt.savefig('../figures/rmst1_ko_vector_field.pdf', dpi=150)
plt.show()


# =============================================================
# SAVE ORACLE OBJECT FOR FURTHER ANALYSIS
# =============================================================

#oracle.to_hdf5("../data/oracle_rmst1_ko.celloracle.hdf5")